# Assignment in short
We have a npy file with all the data. 3 columns, User ID, Movie ID, rating.
We need to make a 2D matrix with users as rows and movies as columns. Cells should be 0 or 1 depending on if the movie has been rated by the user. 
To find similar users, LSH should be used to find candidates to be similar. For the candidates, Jaccard similarity can be used to find most similar users. 

## Step by Step
1. Load in data
2. Reform data
3. Create signatures for users
4. Split signatures into bands
5. Put similar users in same bucket
6. Compute Jaccard similarity on similar users


LSH(locality sensitive hashing) is a technique for quickly finding similar items in large datasets without comparing everything. Similar items are likely to get the same hash value. LSH uses special hash functions that cluster similar things together.
Each user is represented as the set of movies that they rated. minhash signature is its compressed representation.
Signatures are split up into 'bands', each band is split up into rows. If two users have identical rows in at least one band, they might be similar users.

for these 'might be similar' users, compute jaccard similarity to find pairs of most similar users.

### 1. Load in data
Check if file exists, check if file has format as expected, parse columns. get number of users and number of movies

In [9]:
# load user_movie_rating.npy and parse columns

import numpy as np
import os

# check if file exists
user_movie_rating_path = "user_movie_rating.npy"
if not os.path.exists(user_movie_rating_path):
    raise FileNotFoundError(f"{user_movie_rating_path} not found in current directory")

# load data and raise error if data is not as expected
data = np.load(user_movie_rating_path, mmap_mode='r') # mmap_mode='r' for large files
if data.ndim != 2 or data.shape[1] != 3:
    raise ValueError("Expected a 2D array with 3 columns: user_id, movie_id, rating")

# parse columns
users = data[:, 0].astype(int)
movies = data[:, 1].astype(int)
ratings = data[:, 2].astype(int)

#! at the end the specific rating does not matter for building the user-item matrix

n_users = users.max()
n_movies = movies.max()

print(f"Loaded {data.shape[0]} interactions")
print(f"Users: {n_users} ")
print(f"Movies: {n_movies} ")

# example: show first 10 raw rows and their mapped indices
print("First 10 raw rows (user_id, movie_id, rating):")
print(data[:10])



Loaded 65225506 interactions
Users: 103703 
Movies: 17770 
First 10 raw rows (user_id, movie_id, rating):
[[  1  30   3]
 [  1 157   3]
 [  1 173   4]
 [  1 175   5]
 [  1 191   2]
 [  1 197   3]
 [  1 241   3]
 [  1 295   4]
 [  1 299   3]
 [  1 329   4]]


## 2. Reform data
Data should be stores in a sparse scipy matrix. In below block, different formats are displayed and best practises are shown. We use COO to create sparse matrix. Then we transform it to CSR to do further calculations on. create list of arrays, each array contains the movie indices rated by that user

# what sparse matrix storage scheme to use
| Format | Matrix × Vector | Get Item | Fancy Get | Set Item | Fancy Set | Solvers | Notes |
|--------|------------------|----------|-----------|----------|-----------|---------|--------|
| **DIA** | sparsetools | . | . | . | . | iterative | has data array, specialized |
| **LIL** | via CSR | yes | yes | yes | yes | iterative | arithmetics via CSR, incremental construction |
| **DOK** | python | yes | one axis only | yes | yes | iterative | O(1) item access, incremental construction |
| **COO** | sparsetools | . | . | . | . | iterative | has data array, facilitates fast conversion |
| **CSR** | sparsetools | yes | yes | slow | . | any | has data array, fast row-wise ops |
| **CSC** | sparsetools | yes | yes | slow | . | any | has data array, fast column-wise ops |
| **BSR** | sparsetools | . | . | . | . | specialized | has data array, specialized |

For LSH we want fast row access because we create hash values from users(set of movies they rated) CSR is good for this
For the cadidates to be similar, we need to calculate jaccard similarity. we need to look at the intersections of the two movie sets of the users. 

To build the sparse matrix COO LIL and DOK are mostly used. 


In [10]:
# create a sparse user-item rating matrix

from scipy.sparse import coo_matrix

#scipy uses ID's starting from 0
users -= 1
movies -= 1

data_values = np.ones_like(users, dtype=np.uint8) # rating presence indicator, ratings is ignored
coo = coo_matrix((data_values, (users, movies)), shape=(n_users, n_movies), dtype=bool)
print(f"Sparse rating matrix shape: {coo.shape}, nnz={coo.nnz}")
del users, movies, ratings, data
csr = coo.tocsr()
del coo

#print some csr data
print(f"CSR matrix shape: {csr.shape}, nnz={csr.nnz}")
print(f"CSR matrix data sample (first 10 entries): {csr.data[:40]}")

# create list of arrays, each array contains the movie indices rated by that user
user_movie_lists = [
    csr.indices[csr.indptr[i]:csr.indptr[i+1]]
    for i in range(n_users)
]

Sparse rating matrix shape: (103703, 17770), nnz=65225506
CSR matrix shape: (103703, 17770), nnz=65225506
CSR matrix data sample (first 10 entries): [ True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True]


## 3. Create signatures for users
Set number of has functions/permutations, bands and rows. Create random permutations, create signature matrix; k signatures per user.

In [11]:
from tqdm import tqdm #! remove eventually because we can only use numpy and scipy libraries

k = 120 # number of hash functions (permutations)
bands = 30
rows = k // bands # rows is now 4

# create k random permutations of movie indices
permutations = np.array([
    np.random.permutation(n_movies)
    for a in range(k)
], dtype=np.int32)

# initialize signature matrix
signatures = np.empty(shape=(n_users, k), dtype=np.int32)

# loop over each permutation and compute minhash signatures
for j in tqdm(range(k)): #! tqdm to show progress bar. remove in final version
    perm = permutations[j]
    # compute: min perm[movie] for each user
    signatures[:, j] = np.array([
        perm[movies].min() # mishash value for this user and this permutation
        for movies in user_movie_lists
    ])

print(signatures[0:5, :10])  # print first 5 users first 10 signature values
print(signatures.shape)


100%|██████████| 120/120 [00:45<00:00,  2.64it/s]

[[47 12  9  0 40  8 22 43 38 53]
 [32 10  9 38 21 37 12  0 22 53]
 [16 12  2  7 21  8 22 41 54 50]
 [16  2  9 32 21 11 27 41  3 41]
 [63 10  9  7 21  0 14 91 24 22]]
(103703, 120)


## 4. Split signatures into bands


In [12]:
banded_signatures = signatures.reshape(n_users, bands, rows)
print(banded_signatures.shape)
print("User 0, band 0:", banded_signatures[0, 0, :])
print("User 0, band 1:", banded_signatures[0, 1, :])

(103703, 30, 4)
User 0, band 0: [47 12  9  0]
User 0, band 1: [40  8 22 43]


## 5. put similar users in same bucket
Two users go into the same bucket for band b if their 4 minhash signatures in that band are IDENTICAL.
Each band is a vector of 4 integers. We hash this value to 1 integer. users whose band vectors hash to the same integer go in the same bucket and are canidates to be similar.


In [ ]:
import numpy as np

# choose a random hash multiplier per row (for stable hashing)
multipliers = np.random.randint(1, 2**31 - 1, size=rows, dtype=np.int64)

band_buckets = []   # list of: list of buckets, each bucket is a numpy array of user IDs

for b in range(bands):

    # Shape (n_users, rows)
    band = banded_signatures[:, b, :]

    # Compute hash per user
    band_hashes = (band.astype(np.int64) * multipliers).sum(axis=1)

    # Sort users by hash so equal-hash users become adjacent
    order = np.argsort(band_hashes)
    sorted_hashes = band_hashes[order]

    # Find boundaries where the hash value changes
    changes = np.where(sorted_hashes[1:] != sorted_hashes[:-1])[0] + 1

    # Split into groups (buckets)
    groups = np.split(order, changes)

    # Keep buckets only if they contain at least 2 users
    buckets = [g for g in groups if len(g) >= 2]

    band_buckets.append(buckets)

print("Example buckets in band 0:", band_buckets[0][:5])


Example buckets in band 0: [array([45696,  7358, 25497, 70061, 38799], dtype=int64), array([38031, 24118], dtype=int64), array([18112, 93689, 82391, 16528, 34514, 93349, 25274, 20734],
      dtype=int64), array([12245, 14269, 79094, 12996], dtype=int64), array([21293, 30715], dtype=int64)]


get all unique unordered pairs of users that appear together in at least one bucket

In [13]:
candidate_pairs = set()

for buckets in band_buckets:
    for bucket in buckets:
        bucket = np.asarray(bucket)
        bucket_size = bucket.size
        if bucket_size < 2 or bucket_size > 50:
            continue
        # Generate all unordered pairs inside the bucket
        i = np.repeat(np.arange(bucket_size - 1), np.arange(bucket_size - 1, 0, -1))
        j = np.concatenate([np.arange(x + 1, bucket_size) for x in range(bucket_size - 1)])
        pairs = np.stack((bucket[i], bucket[j]), axis=1)

        for u, v in pairs:
            if u < v:
                candidate_pairs.add((u, v))

candidate_pairs = np.array(list(candidate_pairs), dtype=np.int32)
print("Number of candidate pairs:", candidate_pairs.shape[0])


    


Number of candidate pairs: 4110921


## 6. Compute Jaccard similarity on similar users

In [15]:
THRESHOLD = 0.5  # Jaccard similarity threshold

def jaccard(u_items, v_items):
    # intersection size
    inter = np.intersect1d(u_items, v_items, assume_unique=True).size
    # union size
    union = u_items.size + v_items.size - inter
    return inter / union if union > 0 else 0.0

# create or clear output file
open("similar_users.txt", "w").close()

for u, v in candidate_pairs:
    sim = jaccard(user_movie_lists[u], user_movie_lists[v])
    if sim > THRESHOLD:
        with open("similar_users.txt", "a") as f:
            # write u1,u2 to file
            f.write(f"{u},{v}\n")
